# Bespoke Bayesian Optimisation — BBO Capstone Project
**Imperial College Business School | Executive Education**

---

## Project Overview

This notebook implements a **bespoke Bayesian Optimisation (BO) pipeline** for the BBO capstone project.
The goal is to maximise eight unknown black-box functions by submitting one query per function per week,
using a Gaussian Process (GP) surrogate model to guide the search.

### The eight functions simulate:
| Function | Scenario | Dimensions | Initial Points |
|----------|----------|------------|----------------|
| F1 | Contamination source detection | 2D | 10 |
| F2 | Noisy log-likelihood maximisation | 2D | 10 |
| F3 | Drug side-effect minimisation | 3D | 15 |
| F4 | Warehouse ML hyperparameter tuning | 4D | 30 |
| F5 | Chemical process yield (unimodal) | 4D | 20 |
| F6 | Cake recipe optimisation | 5D | 20 |
| F7 | ML model hyperparameter tuning (6 params) | 6D | 30 |
| F8 | ML model hyperparameter tuning (8 params) | 8D | 40 |

### Final acquisition strategy per function:
| Function | Strategy | Rationale |
|----------|----------|-----------|
| F1 | Localised EI | GP misfires globally; constrain to ±0.010 box around known best |
| F2 | UCB (κ=0.8) | Exploit confirmed best at (0.697, 0.321) |
| F3 | UCB (κ=2.0) | Continued exploration |
| F4 | EI | Exploit W11 breakthrough at +0.566 |
| F5 | Manual override | True optimum confirmed at (1,1,1,1) = 8662.48 |
| F6 | EI | Three TS regressions — revert to exploitation |
| F7 | EI | Consistent improver — unchanged throughout |
| F8 | TuRBO | Trust-region BO for 8D — global GP unreliable |

### References
- [1] Hennig & Schuler (2012) — Entropy Search. JMLR 13, 1809–1837.
- [2] Eriksson et al. (2019) — TuRBO. NeurIPS 2019.
- [3] Rasmussen & Williams (2006) — Gaussian Processes for Machine Learning.
- [4] OpenML benchmark datasets.

---

## Cell 1 — Install Dependencies

Run this cell once to install all required packages.

In [ ]:
# Install required packages
# Run this cell once; restart the kernel afterwards if needed
!pip install botorch gpytorch torch numpy matplotlib scipy --quiet

## Cell 2 — Imports

In [ ]:
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import UpperConfidenceBound, ExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.models.transforms.outcome import Standardize
from botorch.generation import MaxPosteriorSampling

from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.constraints import Interval

np.set_printoptions(linewidth=np.inf)
print("All imports successful.")

## Cell 3 — Configuration

**Set `FUNCTION_NO` (1–8) and `WEEK_NO` before running.**

- `FUNCTION_NO`: which of the 8 functions to optimise
- `WEEK_NO`: the current week number (controls how many past queries are loaded)

In [ ]:
# ============================================================
# USER CONFIGURATION — update these two values before running
# ============================================================

FUNCTION_NO = 1   # <-- change to 1–8
WEEK_NO     = 13  # <-- change to current week number

# ============================================================
# PER-FUNCTION KERNEL CONFIGURATION
# ============================================================
# Length-scale bounds control how smooth the GP surface is.
# Tight bounds (small values) = GP is sensitive to sharp changes.
# Wide bounds (large values)  = GP assumes a smoother surface.

LENGTH_SCALE_BOUNDS = {
    1: (0.005, 0.05),  # Very tight — F1 has a near-singular sharp peak
    2: (0.05,  0.5),   # Moderate — 2D log-likelihood
    3: (0.1,   1.0),   # Wider — smooth 3D drug function
    4: (0.05,  0.5),   # Tighter — jagged 4D warehouse function
    5: (0.1,   1.5),   # Very wide — smooth unimodal chemical yield
    6: (0.05,  0.75),  # Moderate — 5D cake recipe
    7: (0.05,  1.0),   # ARD important — 6D HP tuning [Ref 4]
    8: (0.05,  1.0),   # TuRBO manages local scales — 8D HP tuning [Ref 2]
}[FUNCTION_NO]

# ============================================================
# Matérn smoothness parameter ν
# ν=1.5 → once-differentiable (jagged/sharp functions)
# ν=2.5 → twice-differentiable (smooth functions)
# ============================================================

NU_SMOOTHNESS = {
    1: 1.5,  # Sharp peak
    2: 1.5,  # Moderate sharpness
    3: 2.5,  # Smooth
    4: 1.5,  # Jagged landscape
    5: 2.5,  # Smooth unimodal
    6: 2.5,  # Smooth multi-dim
    7: 2.5,  # Smooth HP surface
    8: 2.5,  # Smooth; TuRBO handles locality
}[FUNCTION_NO]

# ============================================================
# ACQUISITION STRATEGY per function
# ============================================================
# 'localised_ei' — EI constrained to small box (F1)
# 'ucb'          — Upper Confidence Bound (F2, F3)
# 'ei'           — Expected Improvement (F4, F6, F7)
# 'override'     — Manual hard override (F5)
# 'turbo'        — Trust-Region BO (F8)

ACQUISITION_STRATEGY = {
    1: 'localised_ei',  # EI within ±0.010 box around known best
    2: 'ucb',           # UCB κ=0.8 — exploit confirmed best
    3: 'ucb',           # UCB κ=2.0 — continued exploration
    4: 'ei',            # EI — exploit W11 breakthrough (+0.566)
    5: 'override',      # Manual override — true optimum at (1,1,1,1)
    6: 'ei',            # EI — three TS regressions; revert to exploitation
    7: 'ei',            # EI — consistent improver throughout
    8: 'turbo',         # TuRBO — global GP unreliable in 8D [Ref 2]
}[FUNCTION_NO]

# UCB kappa values (only used for F2 and F3)
UCB_KAPPA = {
    2: 0.8,  # Reduced from 3.0 → tighter exploitation
    3: 2.0,  # Moderate exploration
}

# ============================================================
# NOISE FLOOR CONSTRAINTS
# Prevents the GP from absorbing genuine signal as noise.
# Tight constraints for sharp/jagged functions (F1, F4).
# ============================================================

NOISE_CONSTRAINT = {
    1: Interval(1e-6, 1e-3),  # Tight — sharp F1
    2: Interval(1e-5, 1e-1),  # Moderate
    3: Interval(1e-5, 1e-1),  # Moderate
    4: Interval(1e-5, 1e-2),  # Tight — jagged F4
    5: Interval(1e-5, 1e-1),  # Moderate
    6: Interval(1e-5, 1e-1),  # Moderate
    7: Interval(1e-5, 1e-1),  # Moderate
    8: Interval(1e-5, 1e-1),  # Moderate
}[FUNCTION_NO]

# ============================================================
# MANUAL OVERRIDES
# ============================================================

# F1: Localised EI search box — ±0.010 around known best (0.703500, 0.716000)
# EI = 0 at all observed points, so this guarantees a new query.
F1_LOCALISED_BOUNDS = torch.tensor(
    [[0.6935, 0.7060],   # lower bounds [x1_lb, x2_lb]
     [0.7135, 0.7260]],  # upper bounds [x1_ub, x2_ub]
    dtype=torch.float64
)

# F5: Boundary override — confirmed optimum is (1,1,1,1) = 8662.48
# (0.99,0.99,0.99,0.99) already submitted in W10; use new point.
F5_BOUNDARY_OVERRIDE = np.array([0.980, 0.980, 0.990, 0.990])

# ============================================================
# TuRBO HYPERPARAMETERS (F8 only)
# Controls the trust region (TR) that restricts where F8 is queried.
# The TR shrinks on failure and grows on success. [Ref 2]
# ============================================================

TURBO_LENGTH_INIT    = 0.4   # Initial TR side length
TURBO_LENGTH_MIN     = 0.01  # Minimum TR length before reset
TURBO_LENGTH_MAX     = 1.6   # Maximum TR length
TURBO_SUCCESS_TOL    = 3     # Successes before TR expands
TURBO_FAILURE_TOL    = 4     # Failures before TR shrinks
TURBO_SUCCESS_FACTOR = 2.0   # TR expansion multiplier
TURBO_FAILURE_FACTOR = 0.5   # TR contraction multiplier

print(f"Configuration set: Function {FUNCTION_NO}, Week {WEEK_NO}")
print(f"Acquisition strategy : {ACQUISITION_STRATEGY.upper()}")
print(f"Matérn ν             : {NU_SMOOTHNESS}")
print(f"Length-scale bounds  : {LENGTH_SCALE_BOUNDS}")

## Cell 4 — Data Loading

Loads the initial dataset and appends all weekly query results up to `WEEK_NO - 1`.

**Expected file structure:**
```
./initial_data/function_{N}/initial_inputs.npy
./initial_data/function_{N}/initial_outputs.npy
./week{k}_inputs.txt    ← one line per function, hyphen-separated
./week{k}_outputs.txt   ← one line per function
```

In [ ]:
def read_data(function_no, week_no, debug=False):
    """
    Load the initial dataset and append all weekly query results.

    Parameters
    ----------
    function_no : int
        Which function to load (1–8).
    week_no : int
        Current week number. Loads weeks 1 through week_no-1.
    debug : bool
        If True, prints all data points.

    Returns
    -------
    X : np.ndarray, shape (n, d)
        All input points observed so far.
    y : np.ndarray, shape (n,)
        Corresponding output values.
    """
    # Load initial dataset
    X = np.load(f"./initial_data/function_{function_no}/initial_inputs.npy")
    y = np.load(f"./initial_data/function_{function_no}/initial_outputs.npy")

    # Append weekly queries
    for week in range(1, week_no):
        with open(f"./week{week}_inputs.txt", 'r') as f:
            line = f.readlines()[function_no - 1]
            new_x = np.array([float(v) for v in line.strip().split('-')])
            X = np.vstack((X, new_x))
        with open(f"./week{week}_outputs.txt", 'r') as f:
            line = f.readlines()[function_no - 1]
            new_y = float(line.strip())
            y = np.append(y, new_y)

    if debug:
        print(f"\nFunction {function_no} — {len(y)} data points total")
        print(f"Input shape : {X.shape}")
        print(f"Output range: {y.min():.4f} to {y.max():.4f}")
        print(f"Best output : {y.max():.6f} at point {np.argmax(y)+1}")

    return X, y


# Load data for the selected function
X, y = read_data(FUNCTION_NO, WEEK_NO, debug=True)

# Show ranked top-5 results
print("\n--- Top 5 observed points ---")
ranked = np.argsort(y)[::-1]
for rank in range(min(5, len(y))):
    i = ranked[rank]
    coords = '-'.join(f'{v:.6f}' for v in X[i])
    print(f"  Rank {rank+1}: [{coords}] → {y[i]:.6f}")

## Cell 5 — GP Model Classes

Two model classes:
- **`BoTorchGP`** — standard GP with function-specific acquisition (F1–F7)
- **`TuRBoGP`** — trust-region GP for high-dimensional F8 [Ref 2]

In [ ]:
# ===========================================================
# BASE CLASS
# Defines the interface all GP models must implement.
# ===========================================================

class GPModel:
    """Abstract base class for GP surrogate models."""
    N = None

    def fit(self, X, y, debug=False):          raise NotImplementedError
    def evaluate(self, X, grid=False):         raise NotImplementedError
    def evaluate_acquisition(self, X, grid=False): raise NotImplementedError
    def find_acquisition_max(self, debug=False): raise NotImplementedError

    @staticmethod
    def _build_gp(X_np, y_np):
        """
        Build a BoTorch SingleTaskGP with:
        - Matérn kernel (ARD enabled — one length-scale per dimension)
        - ScaleKernel wrapper (learns output variance σ_f²)
        - Standardize transform (zero-mean normalisation — critical for
          functions with all-negative outputs like F3 and F6)
        - Per-function noise constraint
        """
        matern = MaternKernel(
            nu=NU_SMOOTHNESS,
            ard_num_dims=X_np.shape[1],
            lengthscale_constraint=Interval(*LENGTH_SCALE_BOUNDS)
        )
        covar_module = ScaleKernel(matern)
        gp = SingleTaskGP(
            torch.tensor(X_np, dtype=torch.float64),
            torch.tensor(y_np, dtype=torch.float64).unsqueeze(-1),
            covar_module=covar_module,
            outcome_transform=Standardize(m=1),
            train_Yvar=None,
        )
        gp.likelihood.noise_covar.register_constraint("raw_noise", NOISE_CONSTRAINT)
        return gp


# ===========================================================
# BoTorchGP — used for Functions 1–7
# ===========================================================

class BoTorchGP(GPModel):
    """
    Standard GP surrogate with function-specific acquisition strategy.
    Supports: localised_ei, ucb, ei, override, thompson.
    """

    def fit(self, X, y, debug=False):
        """Fit the GP by maximising the marginal log-likelihood (MLL)."""
        self.N    = X.shape[1]
        self.X_np = X
        self.y_np = y
        self.gp   = self._build_gp(X, y)
        self.gp.train()
        mll = ExactMarginalLogLikelihood(self.gp.likelihood, self.gp)
        fit_gpytorch_mll(mll)
        self.gp.eval()
        if debug:
            print(f"\nGP fitted [{ACQUISITION_STRATEGY.upper()}]")
            print(f"  Output scale σ_f² : {self.gp.covar_module.outputscale.item():.4f}")
            ls = self.gp.covar_module.base_kernel.lengthscale.detach().numpy().ravel()
            print(f"  Length scales ℓ   : {np.round(ls, 4)}")
            print(f"  Noise σ_n²        : {self.gp.likelihood.noise.item():.6f}")

    def evaluate(self, X, grid=False):
        """Return posterior mean and std at input points X."""
        if grid:
            shape = X[0].shape
            X = np.column_stack([m.ravel() for m in X])
        with torch.no_grad():
            post = self.gp.posterior(torch.tensor(X, dtype=torch.float64))
            mean = post.mean.detach().numpy().ravel()
            std  = post.variance.sqrt().detach().numpy().ravel()
        if grid:
            mean, std = mean.reshape(*shape), std.reshape(*shape)
        return mean, std

    def _build_acquisition(self):
        """
        Select and return the acquisition function object.

        EI (Expected Improvement): queries where expected gain over
            current best f* is highest. Naturally balances exploration
            and exploitation without a tunable parameter.

        UCB (Upper Confidence Bound): μ(x) + κ·σ(x). Higher κ = more
            exploration. Lower κ = more exploitation.

        Thompson Sampling: draws a random function from the posterior
            and returns its argmax. Introduces stochastic diversity.
        """
        if ACQUISITION_STRATEGY in ('ei', 'localised_ei', 'override'):
            best_f = torch.tensor(self.y_np.max(), dtype=torch.float64)
            return ExpectedImprovement(model=self.gp, best_f=best_f)
        elif ACQUISITION_STRATEGY == 'ucb':
            kappa = UCB_KAPPA[FUNCTION_NO]
            return UpperConfidenceBound(model=self.gp, beta=kappa ** 2)
        elif ACQUISITION_STRATEGY == 'thompson':
            return None  # handled separately
        else:
            raise ValueError(f"Unknown strategy: {ACQUISITION_STRATEGY}")

    def evaluate_acquisition(self, X, grid=False):
        """Evaluate acquisition function at X (for plotting)."""
        if grid:
            shape = X[0].shape
            X = np.column_stack([m.ravel() for m in X])
        if ACQUISITION_STRATEGY == 'thompson':
            mean, std = self.evaluate(X.reshape(-1, self.N), grid=False)
            y = mean + 2.0 * std
            return y.reshape(*shape) if grid else y
        acq = self._build_acquisition()
        with torch.no_grad():
            y = acq(torch.tensor(X, dtype=torch.float64).unsqueeze(1)).detach().numpy().ravel()
        return y.reshape(*shape) if grid else y

    def find_acquisition_max(self, debug=False):
        """
        Find the point that maximises the acquisition function.

        Special cases:
        - F1: localised EI within ±0.010 box around (0.703500, 0.716000)
        - F5: hard override to boundary neighbourhood point
        - F4 (thompson): MaxPosteriorSampling over 2048 candidates
        - All others: optimize_acqf with 512 raw samples + 32 restarts
        """

        # F1: Localised EI — constrain search to known peak neighbourhood
        if FUNCTION_NO == 1:
            acq = self._build_acquisition()
            best_x, best_y = optimize_acqf(
                acq_function=acq,
                bounds=F1_LOCALISED_BOUNDS,  # ±0.010 box around (0.7035, 0.7160)
                raw_samples=512,
                num_restarts=32,
                q=1
            )
            best_x = best_x.detach().numpy().ravel()
            best_y = best_y.detach().numpy().item()
            if debug:
                print("\n[F1 LOCALISED EI]")
                print("  Search box: x1∈[0.6935,0.7135], x2∈[0.7060,0.7260]")
                print(f"  Proposed  : {'-'.join(f'{v:.6f}' for v in best_x)}")
                print(f"  EI value  : {best_y:.6f}")
            return best_x, best_y

        # F5: Manual boundary override
        if FUNCTION_NO == 5:
            best_x = F5_BOUNDARY_OVERRIDE
            mean, _ = self.evaluate(best_x.reshape(1, -1))
            if debug:
                print("\n[F5 BOUNDARY OVERRIDE]")
                print(f"  Override  : {'-'.join(f'{v:.6f}' for v in best_x)}")
            return best_x, float(mean[0])

        bounds = torch.stack([
            torch.zeros(self.N, dtype=torch.float64),
            torch.ones(self.N,  dtype=torch.float64)
        ])

        # Thompson Sampling
        if ACQUISITION_STRATEGY == 'thompson':
            candidates = torch.rand(2048, self.N, dtype=torch.float64)
            ts = MaxPosteriorSampling(model=self.gp, replacement=False)
            with torch.no_grad():
                best_x = ts(candidates, num_samples=1).detach().numpy().ravel()
            mean, _ = self.evaluate(best_x.reshape(1, -1))
            if debug:
                print(f"\n[THOMPSON SAMPLING]")
                print(f"  Proposed: {'-'.join(f'{v:.6f}' for v in best_x)}")
            return best_x, float(mean[0])

        # Standard gradient-based optimisation (EI, UCB)
        acq = self._build_acquisition()
        best_x, best_y = optimize_acqf(
            acq_function=acq,
            bounds=bounds,
            raw_samples=512,
            num_restarts=32,
            q=1
        )
        best_x = best_x.detach().numpy().ravel()
        best_y = best_y.detach().numpy().item()
        if debug:
            print(f"\n[{ACQUISITION_STRATEGY.upper()}]")
            print(f"  Proposed  : {'-'.join(f'{v:.6f}' for v in best_x)}")
            print(f"  Acq value : {best_y:.6f}")
        return best_x, best_y


# ===========================================================
# TuRBoGP — used exclusively for Function 8
# ===========================================================

class TuRBoGP(GPModel):
    """
    Trust-Region Bayesian Optimisation for 8D Function 8. [Ref 2]

    Standard global BO fails in 8D because:
    (a) The GP covariance matrix becomes ill-conditioned.
    (b) The acquisition function has exponentially many local maxima.
    (c) The GP is overconfident in unexplored regions.

    TuRBO restricts queries to a trust region (TR) centred on the
    current best point. The TR shrinks on failure and grows on success,
    providing adaptive locality. Thompson Sampling is used within the TR.
    """

    def __init__(self):
        self.tr_length       = TURBO_LENGTH_INIT
        self.success_counter = 0
        self.failure_counter = 0
        self.best_y          = -np.inf

    def fit(self, X, y, debug=False):
        """Fit GP on all data; TR restricts where we query, not how we fit."""
        self.N      = X.shape[1]
        self.X_np   = X
        self.y_np   = y
        self.best_y = float(y.max())
        self.gp     = self._build_gp(X, y)
        self.gp.train()
        mll = ExactMarginalLogLikelihood(self.gp.likelihood, self.gp)
        fit_gpytorch_mll(mll)
        self.gp.eval()
        if debug:
            ls = self.gp.covar_module.base_kernel.lengthscale.detach().numpy().ravel()
            print(f"\n[TuRBO] GP fitted | TR length: {self.tr_length:.4f} | Best y: {self.best_y:.4f}")
            print(f"  Length scales: {np.round(ls, 4)}")

    def _get_tr_bounds(self):
        """
        Compute anisotropic trust region bounds centred on current best.
        Dimensions with longer length-scales get wider TR half-widths,
        reflecting that the GP is less sensitive there.
        """
        best_idx   = np.argmax(self.y_np)
        x_center   = self.X_np[best_idx]
        ls         = self.gp.covar_module.base_kernel.lengthscale.detach().numpy().ravel()
        weights    = ls / ls.max()           # normalise: longest dim gets full width
        half_width = (self.tr_length / 2.0) * weights
        lb = np.clip(x_center - half_width, 0.0, 1.0)
        ub = np.clip(x_center + half_width, 0.0, 1.0)
        return lb, ub

    def update_tr_state(self, new_y):
        """Update TR length based on whether the latest query improved."""
        if new_y > self.best_y:
            self.success_counter += 1
            self.failure_counter  = 0
            self.best_y = new_y
        else:
            self.failure_counter += 1
            self.success_counter  = 0
        if self.success_counter >= TURBO_SUCCESS_TOL:
            self.tr_length = min(self.tr_length * TURBO_SUCCESS_FACTOR, TURBO_LENGTH_MAX)
            self.success_counter = 0
            print(f"[TuRBO] TR expanded → L = {self.tr_length:.4f}")
        if self.failure_counter >= TURBO_FAILURE_TOL:
            self.tr_length = self.tr_length * TURBO_FAILURE_FACTOR
            self.failure_counter = 0
            print(f"[TuRBO] TR shrunk   → L = {self.tr_length:.4f}")
        if self.tr_length < TURBO_LENGTH_MIN:
            self.tr_length = TURBO_LENGTH_INIT
            print(f"[TuRBO] TR reset    → L = {self.tr_length:.4f}")

    def evaluate(self, X, grid=False):
        if grid:
            shape = X[0].shape
            X = np.column_stack([m.ravel() for m in X])
        with torch.no_grad():
            post = self.gp.posterior(torch.tensor(X, dtype=torch.float64))
            mean = post.mean.detach().numpy().ravel()
            std  = post.variance.sqrt().detach().numpy().ravel()
        if grid:
            mean, std = mean.reshape(*shape), std.reshape(*shape)
        return mean, std

    def evaluate_acquisition(self, X, grid=False):
        """UCB proxy within TR for visualisation (TS has no closed form)."""
        if grid:
            shape = X[0].shape
            X = np.column_stack([m.ravel() for m in X])
        mean, std = self.evaluate(X)
        acq = mean + 2.0 * std
        lb, ub = self._get_tr_bounds()
        inside_tr = np.all((X >= lb) & (X <= ub), axis=1)
        acq = np.where(inside_tr, acq, acq.min())
        return acq.reshape(*shape) if grid else acq

    def find_acquisition_max(self, debug=False):
        """Thompson Sampling within the trust region."""
        lb, ub = self._get_tr_bounds()
        lb_t = torch.tensor(lb, dtype=torch.float64)
        ub_t = torch.tensor(ub, dtype=torch.float64)
        # Sample 2048 candidates uniformly within the TR
        candidates = lb_t + (ub_t - lb_t) * torch.rand(2048, self.N, dtype=torch.float64)
        ts = MaxPosteriorSampling(model=self.gp, replacement=False)
        with torch.no_grad():
            best_x = ts(candidates, num_samples=1).detach().numpy().ravel()
        mean, _ = self.evaluate(best_x.reshape(1, -1))
        if debug:
            print(f"\n[TuRBO + TS] TR length: {self.tr_length:.4f}")
            print(f"  TR bounds : lb={np.round(lb,3)}, ub={np.round(ub,3)}")
            print(f"  Proposed  : {'-'.join(f'{v:.6f}' for v in best_x)}")
        return best_x, float(mean[0])


print("GP model classes defined successfully.")

## Cell 6 — Fit the Model and Generate the Query

This cell fits the GP surrogate to all accumulated data and proposes the next query point.

In [ ]:
# Select the correct model class based on acquisition strategy
if ACQUISITION_STRATEGY == 'turbo':
    model = TuRBoGP()
else:
    model = BoTorchGP()

# Fit the GP surrogate to all accumulated data
model.fit(X, y, debug=True)

# Find the acquisition function maximum — this is the proposed query
proposed_x, acq_value = model.find_acquisition_max(debug=True)

## Cell 7 — Display the Proposed Query

The proposed query is formatted for direct copy-paste into the capstone portal.

In [ ]:
# Format the proposed query for portal submission
portal_string = '-'.join(f'{v:.6f}' for v in proposed_x)

print("=" * 60)
print(f"FUNCTION {FUNCTION_NO} — WEEK {WEEK_NO} QUERY")
print("=" * 60)
print(f"\nProposed input vector:")
for i, v in enumerate(proposed_x):
    print(f"  x{i+1} = {v:.6f}")
print(f"\nPortal submission string:")
print(f"  {portal_string}")
print(f"\nAcquisition value : {acq_value:.6f}")
print(f"Current best      : {y.max():.6f}")
print(f"Strategy          : {ACQUISITION_STRATEGY.upper()}")
print("=" * 60)

## Cell 8 — Visualise the GP Posterior

Plots the GP posterior mean, uncertainty, and acquisition function.
- **2D functions (F1, F2):** 3D surface plot
- **3D functions (F3):** 3D scatter with acquisition overlay
- **4D+ functions (F4–F8):** Parallel coordinates plot

In [ ]:
def plot_gp_results(model, X, y, function_no, week_no, proposed_x):
    """
    Visualise the GP posterior and acquisition function.
    Automatically selects the appropriate plot type based on dimensionality.
    """
    N = X.shape[1]
    n_initial = len(y) - (week_no - 1)  # number of initial data points

    # ------------------------------------------------------------------
    # 1D input → 2D line plot (not used in this capstone but included)
    # ------------------------------------------------------------------
    if N == 1:
        linspace    = np.linspace(0, 1, 200)
        mean, std   = model.evaluate(linspace.reshape(-1, 1))
        acq_vals    = model.evaluate_acquisition(linspace.reshape(-1, 1))
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
        ax1.scatter(X[:n_initial], y[:n_initial], c='black', marker='o', s=40, label='Initial data')
        ax1.scatter(X[n_initial:], y[n_initial:], c='red',   marker='s', s=60, label='Weekly queries')
        ax1.plot(linspace, mean, 'b-', linewidth=2, label='GP mean')
        ax1.fill_between(linspace, mean-std, mean+std, alpha=0.2, label='GP ±1σ')
        ax1.axvline(proposed_x[0], color='green', linestyle='--', label='Proposed query')
        ax1.set_ylabel('Output'); ax1.legend(); ax1.set_title(f'F{function_no} GP Posterior')
        ax2.plot(linspace, acq_vals, 'g-', linewidth=2)
        ax2.axvline(proposed_x[0], color='green', linestyle='--')
        ax2.set_xlabel('x'); ax2.set_ylabel('Acquisition'); ax2.set_title('Acquisition Function')
        plt.tight_layout(); plt.show()

    # ------------------------------------------------------------------
    # 2D input → 3D surface plot
    # ------------------------------------------------------------------
    elif N == 2:
        linspace = np.linspace(0, 1, 30)
        mesh     = np.meshgrid(linspace, linspace, indexing='ij')
        mean, std = model.evaluate(mesh, grid=True)
        acq_vals  = model.evaluate_acquisition(mesh, grid=True)

        fig = plt.figure(figsize=(14, 5))

        # GP posterior mean surface
        ax1 = fig.add_subplot(121, projection='3d')
        ax1.plot_surface(mesh[0], mesh[1], mean, cmap='viridis', alpha=0.7)
        ax1.scatter(X[:n_initial,0], X[:n_initial,1], y[:n_initial], c='black', s=30, depthshade=True)
        ax1.scatter(X[n_initial:,0], X[n_initial:,1], y[n_initial:], c='red',   s=60, marker='s', depthshade=True)
        ax1.scatter([proposed_x[0]], [proposed_x[1]], [mean.max()], c='lime', s=200, marker='*')
        ax1.set_xlabel('x1'); ax1.set_ylabel('x2'); ax1.set_zlabel('y')
        ax1.set_title(f'F{function_no} — GP Posterior Mean [{ACQUISITION_STRATEGY.upper()}]')

        # Acquisition function surface
        ax2 = fig.add_subplot(122, projection='3d')
        ax2.plot_surface(mesh[0], mesh[1], acq_vals, cmap='plasma', alpha=0.7)
        ax2.scatter([proposed_x[0]], [proposed_x[1]], [acq_vals.max()], c='lime', s=200, marker='*', label='Proposed')
        ax2.set_xlabel('x1'); ax2.set_ylabel('x2'); ax2.set_zlabel('Acquisition')
        ax2.set_title('Acquisition Function')

        plt.tight_layout(); plt.show()

    # ------------------------------------------------------------------
    # 3D input → 3D scatter coloured by output value
    # ------------------------------------------------------------------
    elif N == 3:
        fig = plt.figure(figsize=(10, 8))
        ax  = fig.add_subplot(111, projection='3d')
        norm  = plt.Normalize(y.min(), y.max())
        cmap  = plt.cm.rainbow
        sizes = 40 * (y - y.min()) / (y.max() - y.min() + 1e-9) + 10
        sc = ax.scatter(X[:,0], X[:,1], X[:,2], c=y, cmap='rainbow',
                        s=sizes, vmin=y.min(), vmax=y.max(), depthshade=True)
        ax.scatter([proposed_x[0]], [proposed_x[1]], [proposed_x[2]],
                   c='lime', s=300, marker='*', label='Proposed query')
        ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.set_zlabel('x3')
        ax.set_title(f'F{function_no} — Observed Data (colour = output)')
        fig.colorbar(sc, ax=ax, label='Output value')
        ax.legend(); plt.tight_layout(); plt.show()

    # ------------------------------------------------------------------
    # 4D+ input → Parallel coordinates plot
    # ------------------------------------------------------------------
    else:
        fig, ax = plt.subplots(figsize=(12, 6))
        norm = plt.Normalize(y.min(), y.max())
        cmap = plt.cm.rainbow
        x_axis = np.arange(N)

        # Initial data points — thin solid lines
        for i in range(n_initial):
            ax.plot(x_axis, X[i,:], color=cmap(norm(y[i])), linewidth=1, alpha=0.5)

        # Weekly query points — thicker dashed lines
        for i in range(n_initial, len(y)):
            ax.plot(x_axis, X[i,:], color=cmap(norm(y[i])), linewidth=2,
                    linestyle='--', alpha=0.8)

        # Proposed query — thick green dashed line
        ax.plot(x_axis, proposed_x, color='lime', linewidth=3,
                linestyle='--', label='Proposed query (this week)')

        ax.set_xticks(x_axis)
        ax.set_xticklabels([f'x[{d+1}]' for d in range(N)])
        ax.set_ylabel('Input value (normalised to [0,1])')
        ax.set_title(f'F{function_no} — Parallel Coordinates [{ACQUISITION_STRATEGY.upper()}]')
        ax.legend(loc='upper right')

        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        fig.colorbar(sm, ax=ax, label='Output value')
        plt.tight_layout(); plt.show()


# Generate the plot
plot_gp_results(model, X, y, FUNCTION_NO, WEEK_NO, proposed_x)

## Cell 9 — Results Summary

Displays the all-time best outputs across all functions based on the complete query history.

In [ ]:
# All-time best outputs (final capstone results)
all_time_bests = {
    1: {'output': 1.81e-9,   'input': '0.703500-0.716000',                                          'week': 2},
    2: {'output': 0.8135,    'input': '0.701523-0.379257',                                          'week': 13},
    3: {'output': -0.03484,  'input': '0.492581-0.611593-0.340176',                                 'week': 0},
    4: {'output': 0.56613,   'input': '0.418286-0.384490-0.407242-0.350393',                        'week': 11},
    5: {'output': 8662.48,   'input': '1.000000-1.000000-1.000000-1.000000',                        'week': 8},
    6: {'output': -0.10788,  'input': '0.442035-0.373720-0.624334-0.734770-0.090751',               'week': 13},
    7: {'output': 3.0560,    'input': '0.174678-0.000000-0.504524-0.280666-0.306003-0.661563',      'week': 13},
    8: {'output': 9.9668,    'input': '0.120073-0.227667-0.153431-0.175346-0.955445-0.396587-0.208539-0.708051', 'week': 12},
}

print("=" * 75)
print("FINAL CAPSTONE RESULTS — ALL-TIME BESTS")
print("=" * 75)
print(f"{'F':>2}  {'Best Output':>15}  {'Week':>5}  Best Input")
print("-" * 75)
for fn, data in all_time_bests.items():
    week_str = 'Initial' if data['week'] == 0 else f"W{data['week']}"
    print(f"F{fn}  {data['output']:>15.6g}  {week_str:>6}  {data['input']}")
print("=" * 75)
print("\nKey achievements:")
print("  F5: +13,400% improvement (boundary corner discovery via UCB κ=1000)")
print("  F7: +124% improvement (consistent EI with low x1/x2 signal)")
print("  F4: +114% improvement (Thompson Sampling discovered positive region)")
print("  F3: 0% improvement (original data best never beaten — GP mismatch)")